# Procesamiento de grafos en Spark con GraphFrames

Recordemos que GraphFrames no es parte de Spark. Es un paquete externo que permite procesar grafos usando Spark DataFrames en lugar de los RDD menos cómodos, como hacía el antiguo módulo GraphX que sí es parte de Spark. Se espera que GraphFrames se incorpore oficialmente a Spark en el futuro cercano.

Mientras tanto, primero debemos instalar el paquete GraphFrames para python, que es solo un wrapper de Python para el código Scala que contiene la implementación realmente paralela y distribuida. Podemos instalar el paquete de Python a través de `pip install` o utilizando la funcionalidad de Databricks para instalar paquetes del repositorio pypi. Vamos a la sección "Cómputo", hacemos click en nuestro cluster, buscamos la pestaña superior "Bibliotecas" -> botón azul Instalar nueva -> Pypi y escribimos `graphframes-latest==0.8.3`.

Para la implementación de Scala, tenemos que indicar el paquete GraphFrames como dependencia de Maven (Java/Scala). Esta biblioteca debe estar presente en todos los workers donde haya executors así que la podemos instalar también utilizando la funcionalidad de Databricks. Vamos a Cómputo, click en nuestro cluster, pestaña "Bibliotecas" -> botón Instalar nueva -> Maven y escribimos `graphframes:graphframes:0.8.1-spark3.0-s_2.12`.

Esto descargará la implementación GraphFrames para Scala sobre la marcha. El paquete de Python se basa por completo en dicho código Scala y generará un error si no se encuentra. 

In [0]:
from pyspark.sql import functions as F
from graphframes import GraphFrame

flightsDF = spark.read.option("header", "true")\
                      .option("inferSchema", "true")\
                      .csv("abfss://datos@cursosparkucm.dfs.core.windows.net/flights-jan-apr-2018.csv")

verticesDF = flightsDF.select(F.col("Origin").alias("id")).distinct().cache()

edgesDF = flightsDF.withColumnRenamed("Origin", "src")\
                   .withColumnRenamed("Dest", "dst")\
                   .select("src", "dst", "Distance")\
                   .distinct()\
                   .cache() # select a few columns just to keep things simple
                
graph = GraphFrame(verticesDF, edgesDF)

Vamos a calcular los aeropuertos con más vuelos como aquellos con el grado más alto en su vértice (suma de ambos). El aeropuerto que queda en primer lugar es ORD (O'Hare International Airport) de Chicago.

In [0]:
graph.degrees.orderBy(F.col("degree").desc()).show(3)

+---+------+
| id|degree|
+---+------+
|ORD|   330|
|ATL|   321|
|DFW|   314|
+---+------+
only showing top 3 rows



In [0]:
graph.inDegrees.orderBy(F.col("inDegree").desc()).show(3)

+---+--------+
| id|inDegree|
+---+--------+
|ORD|     165|
|ATL|     161|
|DFW|     157|
+---+--------+
only showing top 3 rows



In [0]:
graph.outDegrees.orderBy(F.col("outDegree").desc()).show(3)

+---+---------+
| id|outDegree|
+---+---------+
|ORD|      165|
|ATL|      160|
|DFW|      157|
+---+---------+
only showing top 3 rows



## PageRank para determinar la importancia global de los vértices

Si ejecutamos PageRank, que es un poco más sofisticado que contar simplemente el número de aristas que llegan o salen de cada vértice, vemos que el aeropuerto con más importancia considerando globalmente todas las conexiones con el resto de la red no es ORD sino DFW (Dallas Fort Worth International Airport).

In [0]:
### ADVERTENCIA: pageRank tarda unos minutos en ejecutarse. Descomentar la siguiente línea si realmente queremos ejecutarlo :-)

ranks = graph.pageRank(resetProbability=0.15, maxIter=10)

In [0]:
ranks.vertices.orderBy(F.col("pagerank").desc()).show()

+---+------------------+
| id|          pagerank|
+---+------------------+
|DFW|10.473812993687389|
|ORD|10.260145353782423|
|ATL| 9.430386741956672|
|DEN| 8.853062782143194|
|MSP| 7.554764057993905|
|CLT| 7.244796334346504|
|DTW|   6.2044026817053|
|LAS| 6.105757091251461|
|IAH| 5.961382132435388|
|PHX| 5.365815675762966|
|SEA| 5.316199446419209|
|SLC| 5.044415678853288|
|PHL| 4.882291006095359|
|LAX| 4.771622359389784|
|EWR|  4.53307436045935|
|SFO| 4.503367605949668|
|SFB| 4.292855657463286|
|DCA| 4.282669744929368|
|IAD|3.9875019394408557|
|MCO|3.8383617855552123|
+---+------------------+
only showing top 20 rows



## Componentes conexas

En nuestor grafo es posible llegar a cualquier aeropuerto desde cualquier otro, y por tanto solo hay una componente conexa y la columna `component` (identificador de la componente) tiene el mismo valor para todos los vértices.

**ADVERTENCIA**: la siguiente celda puede consumir entre 5 y 10 minutos.

In [0]:
spark.sparkContext.setCheckpointDir("/dbfs/FileStore/")
conCompResult = graph.connectedComponents(checkpointInterval=1)
conCompResult.display()

id,component
BGM,0
INL,0
PSE,0
MSY,0
PPG,0
GEG,0
BUR,0
SNA,0
GRB,0
GTF,0


## Consultas por estructura: rutas de vuelo entre aeropuertos sin conexión directa

Estamos indicando que queremos encontrar vértices **a**, **b** y **c** para que haya una arista de **a** a **b**, otra de **b** a **c**, pero no de **a** a **c**, por lo tanto, **a** y **c** no están conectados en un solo salto sino que requieren al menos dos. La restricción adicional evita que **a** y **c** sean el mismo vértice, ya que ningún aeropuerto está conectado consigo mismo, por lo que cada aeropuerto cumpliría individualmente la condición respecto a sí mismo.

In [0]:
res = graph\
 .find("(a)-[]->(b); (b)-[]->(c); !(a)-[]->(c)")\
 .filter("c.id !=a.id")

res.limit(10).display()

/databricks/spark/python/pyspark/sql/dataframe.py:159: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


a,b,c
List(JFK),List(DFW),List(GUC)
List(OAJ),List(ATL),List(RDU)
List(BZN),List(EWR),List(SNA)
List(GUM),List(HNL),List(ANC)
List(RNO),List(SFO),List(IAD)
List(BIL),List(SEA),List(HLN)
List(DFW),List(ORD),List(DBQ)
List(MCI),List(SEA),List(HLN)
List(BDL),List(EWR),List(SNA)
List(CLL),List(IAH),List(SDF)


In [0]:
res.printSchema()

## Caminos más cortos: Breadth-first search

Vamos a encontrar el camino más corto entre dos aeropuertos que no están directamente conectados

<div class="alert alert-danger">
    <b>IMPORTANTE</b>: BFS (Breadth-first search) en Spark calcula el camino mínimo en términos de <b>número de saltos</b> entre dos vértices. No tiene en cuenta el peso de las aristas. Se podría implementar pero de forma personalizada, no con la función bfs().
</div>

Como hay varios caminos que tienen 2 saltos (2 aristas unidas por un único vértice intermedio) entre ABQ y BNA, la función `bfs` devuelve todos ellos.

In [0]:
paths = graph.bfs(fromExpr = "id = 'ABQ'", toExpr= "id = 'BNA'")
paths.display()

/databricks/spark/python/pyspark/sql/dataframe.py:159: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


from,e0,v1,e1,to
List(ABQ),"List(ABQ, IAH, 744.0)",List(IAH),"List(IAH, BNA, 657.0)",List(BNA)
List(ABQ),"List(ABQ, MCI, 718.0)",List(MCI),"List(MCI, BNA, 491.0)",List(BNA)
List(ABQ),"List(ABQ, LAX, 677.0)",List(LAX),"List(LAX, BNA, 1797.0)",List(BNA)
List(ABQ),"List(ABQ, SLC, 493.0)",List(SLC),"List(SLC, BNA, 1404.0)",List(BNA)
List(ABQ),"List(ABQ, PHX, 328.0)",List(PHX),"List(PHX, BNA, 1449.0)",List(BNA)
List(ABQ),"List(ABQ, DAL, 580.0)",List(DAL),"List(DAL, BNA, 623.0)",List(BNA)
List(ABQ),"List(ABQ, SFO, 896.0)",List(SFO),"List(SFO, BNA, 1969.0)",List(BNA)
List(ABQ),"List(ABQ, MDW, 1121.0)",List(MDW),"List(MDW, BNA, 395.0)",List(BNA)
List(ABQ),"List(ABQ, ATL, 1269.0)",List(ATL),"List(ATL, BNA, 214.0)",List(BNA)
List(ABQ),"List(ABQ, ORD, 1118.0)",List(ORD),"List(ORD, BNA, 409.0)",List(BNA)


In [0]:
paths.printSchema()

root
 |-- from: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |-- e0: struct (nullable = false)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |    |-- Distance: double (nullable = true)
 |-- v1: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |-- e1: struct (nullable = false)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |    |-- Distance: double (nullable = true)
 |-- to: struct (nullable = false)
 |    |-- id: string (nullable = true)

